In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Чтение файла
df = pd.read_csv('kdd_train.csv')
df.head()

In [ ]:
rows, columns = df.shape
print(f'Строк: {rows}, Столбцов: {columns}')

In [ ]:
result = df.isnull().sum()
print(f'Всего пропусков: {result}')

In [ ]:
# Пункт 6: Логика заполнения пустых ячеек
for col in df.columns:
    # Проверяем, есть ли пропуски в конкретном столбце
    if df[col].isnull().sum() > 0:
        # Если столбец числовой — берем устойчивую к выбросам медиану
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
        # Если столбец категориальный (текст) — берем самое частое значение (моду)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
print(f'Пропусков после обработки: {df.isnull().sum().sum()}')

In [ ]:
# Пункт 7: Определение и удаление столбцов, не являющихся информативными признаками

# 1. Поиск колонок, где пропусков больше 50%
cols_with_many_nulls = [
    col for col in df.columns
        if df[col].isnull().mean() > 0.5
]

# 2. Поиск колонок-констант (где всего 1 уникальное значение на весь датасет)
constant_cols = [
    col for col in df.columns
        if df[col].nunique() == 1
]

# Объединяем кандидатов на удаление
cols_to_drop = list(set(cols_with_many_nulls + constant_cols))
print(f'Столбцы к удалению: {cols_to_drop}')

# Удаляем эти столбцы из таблицы
df.drop(columns = cols_to_drop, inplace = True)

print(f'Осталось столбцов: {df.shape[1]}')

In [ ]:
# Пункт 8: Разделение на числовые и категориальные признаки

# Забираем только столбцы с числами
df_numeric = df.select_dtypes(include = ['int64', 'float64'])

# Забираем только столбцы с текстом/категориями
df_categorical = df.select_dtypes(include = ['object'])

# Проверяем, сколько столбцов попало в каждый DataFrame
print(f'Числовых признаков: {df_numeric.shape[1]}')
print(f'Категориальных признаков: {df_categorical.shape[1]}')

In [ ]:
# Пункт 9а: Аналитика по количественным признакам
numeric_stats = pd.DataFrame(
    {
        'Минимум': df_numeric.min(),
        'Максимум': df_numeric.max(),
        'Среднее': df_numeric.mean(),
        'Медиана': df_numeric.median(),
        'Дисперсия': df_numeric.var(),
        'Квантиль 0.1': df_numeric.quantile(0.1),
        'Квантиль 0.9': df_numeric.quantile(0.9),
        'Квантиль 0.25': df_numeric.quantile(0.25), 
        'Квантиль 0.75': df_numeric.quantile(0.75),
    }
)

# Сохраняем результат в CSV файл
numeric_stats.to_csv('numeric_stats.csv', encoding='utf-8-sig')

numeric_stats.head(40)

In [ ]:
# Пункт 9b: Аналитика по категориальным признакам
categorical_stats = pd.DataFrame(
    {
        'Уникальных значений': df_categorical.nunique(),
        'Мода': df_categorical.apply(lambda x: x.mode()[0]),
    }
)

# Сохраняем результат в CSV файл
categorical_stats.to_csv('categorical_stats.csv', encoding='utf-8-sig')

# Выводим таблицу целиком (в ней всего 4 строки)
categorical_stats

### Ответ на пункт 10:
**Да, в наборе данных присутствуют категориальные признаки, требующие обязательного преобразования:**

1. `protocol_type` (3 уникальных значения), `service` (70 значений) и `flag` (11 значений):
   * Это номинативные признаки без естественного порядка (нельзя сказать, что tcp > udp).
   * Их необходимо преобразовать с помощью **One-Hot Encoding** (горячее кодирование через `pd.get_dummies` или `OneHotEncoder`), превратив каждое уникальное значение в отдельный бинарный столбец (0 или 1).
   
2. `labels` (целевой признак, 23 уникальных значения):
   * Содержит текстовые названия атак и нормального состояния сети.
   * Для обучения моделей классификации требует кодирования:
     - Либо через **Label Encoding** (каждому типу атаки сопоставляется целое число: normal=0, neptune=1 и т.д.);
     - Либо через **бинаризацию** для задачи бинарной классификации (0 — нормальный трафик, 1 — атака).